### **TABL**

In [4]:
# load packages

import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

from tqdm import tqdm 
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import torch
from torch.utils import data
import torch.nn as nn
import torch.optim as optim
import pickle
import torch.nn.functional as F
from torch.autograd import Variable
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


### **Data**
The dataset in the folder Dataset is the FI-2010 dataset zipped and normalized. 

As in the original paper I used the firs 7 days to train and to validate, and the rest 3 days to do the the testing. 

In [7]:
train_fold = 1


dec_data = np.loadtxt(f'../data/training/Train_Dst_NoAuction_ZScore_CF_{train_fold}.txt')
dec_train = dec_data[:, :]
dec_val = dec_data[:, int(dec_data.shape[1] * 0.80):]
dec_test = np.loadtxt(f'../data/testing/Test_Dst_NoAuction_ZScore_CF_{train_fold}.txt')


h = 5      #if h = 2, then horizon = 50
T = 10      #horizon 
dim = 10
selected_dimention = 144

y_train = dec_train[-h, :].flatten()
y_val = dec_val[-h, :].flatten()
y_test = dec_test[-h, :].flatten()
y_train = y_train[dim-1:] - 1
y_val = y_val[dim-1:] - 1
y_test = y_test[dim-1:] - 1 

dec_train = dec_train[:selected_dimention, :].T
dec_val = dec_val[:selected_dimention, :].T
dec_test = dec_test[:selected_dimention, :].T


In [8]:
y_train.shape

(39503,)

In [9]:
len(dec_val)

7903

In [10]:
print(dec_data.shape)

(149, 39512)


In [12]:
# PiggyBack setup

train_fold = 9



dec_data = np.loadtxt(f'../data/testing/Test_Dst_NoAuction_ZScore_CF_{train_fold-1}.txt')
dec_train = dec_data[:, :]
dec_val = dec_data[:, int(dec_data.shape[1] * 0.80):]
dec_test = np.loadtxt(f'../data/testing/Test_Dst_NoAuction_ZScore_CF_{train_fold}.txt')


h = 5        #if h = 2, then horizon = 50
T = 10      #horizon 
dim = 10
selected_dimention = 144

y_train = dec_train[-h, :].flatten()
y_val = dec_val[-h, :].flatten()
y_test = dec_test[-h, :].flatten()
y_train = y_train[dim-1:] - 1
y_val = y_val[dim-1:] - 1
y_test = y_test[dim-1:] - 1 

dec_train = dec_train[:selected_dimention, :].T
dec_val = dec_val[:selected_dimention, :].T
dec_test = dec_test[:selected_dimention, :].T


In [13]:
class Dataset(data.Dataset):
    """Characterizes a dataset for PyTorch"""
    def __init__(self, x, y, num_classes, dim):
        """Initialization""" 
        self.num_classes = num_classes
        self.dim = dim
        self.x = x   
        self.y = y

        self.length = x.shape[0] - (T/10) -self.dim + 1
        print(self.length)

        x = torch.from_numpy(x)
        self.x = torch.unsqueeze(x, 1)
        self.y = torch.from_numpy(y)

    def __len__(self):
        """Denotes the total number of samples"""
        return int(self.length)

    def __getitem__(self, i):
        input = self.x[i:i+self.dim, :]
        input = input.permute(1, 2, 0)
        input = torch.squeeze(input)

        return input, self.y[i]

In [14]:
#Hyperparameters
batch_size = 256
epochs = 200  
lr = 0.01
num_classes = 3

dataset_val = Dataset(dec_val, y_val, num_classes, dim)
dataset_test = Dataset(dec_test, y_test, num_classes, dim)
dataset_train = Dataset(dec_train, y_train, num_classes, dim)

train_loader = torch.utils.data.DataLoader(dataset=dataset_train, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(dataset=dataset_val, batch_size=batch_size, shuffle=False)
test_loader = torch.utils.data.DataLoader(dataset=dataset_test, batch_size=batch_size, shuffle=False)
train_loader.dataset.x.shape

10425.0
31927.0
52162.0


torch.Size([52172, 1, 144])

In [15]:

all_x = []
all_y = []

In [16]:
for batch in train_loader:
    x, y = batch
    all_x.append(x)
    all_y.append(y)

# Concatenate all batches
all_x = torch.cat(all_x)
all_y = torch.cat(all_y)

In [17]:
all_x.shape

torch.Size([52162, 144, 10])

In [18]:
all_y.shape

torch.Size([52162])

In [19]:
#computing the weights for the weighted cross entropy loss
def compute_weights(y):
  cont_0 = 0
  cont_1 = 0
  cont_2 = 0
  for i in range(y.shape[0]):
    if (y[i] == 0):
      cont_0 += 1
    elif (y[i] == 1):
      cont_1 += 1
    elif (y[i] == 2):
      cont_2 += 2
    else: 
      raise Exception("wrong labels")
  return torch.Tensor([1e6/cont_0, 1e6/cont_1, 1e6/cont_2]).to(device)

y_total = y_train
weights = compute_weights(y_total)